# 🥇 Camada Gold — Analytics SISCAN / 10ª CRS

Transforma as tabelas consolidadas da **Camada Silver** em tabelas fato analíticas no schema `gold`, calculando as métricas exigidas pelas questões de pesquisa do TCC (Q1–Q5): distribuição espacial e evolução temporal, efeito do Outubro Rosa, conformidade da faixa etária, adequabilidade das amostras e proporção de exames alterados.

**Princípios:** preservação de nulos reais (sem `coalesce(x, 0)`), rastreabilidade (`_silver_source`, `_gold_processed_time`) e validação estrita (PK única, sem nulos em colunas obrigatórias) **antes** de cada gravação.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Catálogo e schemas
CATALOG = "`tcc-analise-siscan-10crs`"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"

# 1. Escopo Temporal
ANO_INICIO = 2020
ANO_FIM = 2025

# Garantir schema gold
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}")
print(f"✅ Schema {CATALOG}.{SCHEMA_GOLD} garantido.")


def validar_e_gravar(df, tabela_gold, pk_cols, obrigatorio_cols):
    """Valida PK única e ausência de nulos em colunas obrigatórias ANTES de gravar na Gold."""
    nome = f"{CATALOG}.{SCHEMA_GOLD}.{tabela_gold}"
    n = df.count()

    # 1. Chave primária única
    dups = df.groupBy(pk_cols).count().filter(F.col("count") > 1).count()
    assert dups == 0, f"❌ {tabela_gold}: {dups} grupos de PK duplicada em {pk_cols}"

    # 2. Nulos em colunas obrigatórias (dimensões — métricas podem ser nulas por decisão de negócio)
    for c in obrigatorio_cols:
        nulos = df.filter(F.col(c).isNull()).count()
        assert nulos == 0, f"❌ {tabela_gold}: {nulos} nulos na coluna obrigatória '{c}'"

    # 3. Gravar somente se as validações passarem
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome)
    )
    print(f"✅ {tabela_gold}: {n} linhas gravadas (PK {pk_cols} validada).")
    return n

## 1. GOLD: `fato_exames_temporais_territoriais` (Q1 + Q2)

**Granularidade:** Município × Mês/Ano (`dt_competencia`).

**Métricas (Especificação do Problema Analítico):**
* Quantidade de exames ($Q_{m,t}$)
* Participação Territorial (%) $= \frac{\text{Exames do Município}}{\text{Total 10ª CRS no Período}} \times 100$
* Variação Absoluta $= Q_t - Q_{t-1}$
* Variação Percentual (%) $= \frac{Q_t - Q_{t-1}}{Q_{t-1}} \times 100$

> **Nulos reais preservados:** quando o mês anterior não existe ou a quantidade é ausente na Silver, as variações ficam `NULL` — não são fabricados zeros falsos (Ponto 5 das diretrizes).

In [0]:
# ---------------------------------------------------------------------------
# 1. GOLD: fato_exames_temporais_territoriais (Q1 + Q2)
# ---------------------------------------------------------------------------
df_silver_exames = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.exames_geral")

# Total da regional no mesmo mês/ano (janela por competência)
win_regional = Window.partitionBy("dt_competencia")
# Quantidade do mês anterior (Q_{t-1}) por município
win_lag_muni = Window.partitionBy("codigo_municipio").orderBy("dt_competencia")

df_q1_q2 = (
    df_silver_exames
    .withColumn("cd_ibge", F.lpad(F.col("codigo_municipio").cast("string"), 6, "0"))
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_SILVER}.exames_geral"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .withColumn("total_exames_10crs", F.sum("quantidade").over(win_regional))
    .withColumn(
        "participacao_territorial_pct",
        F.when(F.col("total_exames_10crs") > 0,
               F.round((F.col("quantidade") / F.col("total_exames_10crs")) * 100, 2)),
    )
    # Correção (31/out/2026): o lag só vale se a linha anterior for o mês CALENDRÁRIO
    # imediatamente precedente. Sem essa checagem, set/2022 compararia com jul/2022
    # (Δ de 2 meses) porque ago/2022 não existe na fonte (TABNET omite a coluna —
    # 0 exames na região inteira). Se o mês anterior calendário não existir,
    # quantidade_mes_anterior e as variações ficam NULL (nulo real preservado).
    .withColumn("_dt_lag", F.lag("dt_competencia", 1).over(win_lag_muni))
    .withColumn("_qtde_lag", F.lag("quantidade", 1).over(win_lag_muni))
    .withColumn(
        "quantidade_mes_anterior",
        F.when(F.add_months(F.col("_dt_lag"), 1) == F.col("dt_competencia"), F.col("_qtde_lag")),
    )
    .withColumn("variacao_absoluta", F.col("quantidade") - F.col("quantidade_mes_anterior"))
    .withColumn(
        "variacao_percentual_pct",
        F.when(F.col("quantidade_mes_anterior") > 0,
               F.round(((F.col("quantidade") - F.col("quantidade_mes_anterior"))
                        / F.col("quantidade_mes_anterior")) * 100, 2)),
    )
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano", "mes", "dt_competencia",
        "quantidade", "total_exames_10crs", "participacao_territorial_pct",
        "quantidade_mes_anterior", "variacao_absoluta", "variacao_percentual_pct",
        "_silver_source", "_gold_processed_time",
    )
)

validar_e_gravar(
    df_q1_q2,
    "fato_exames_temporais_territoriais",
    pk_cols=["codigo_municipio", "dt_competencia"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "mes", "dt_competencia"],
)

display(df_q1_q2.orderBy("codigo_municipio", "dt_competencia").limit(12))

## 2. GOLD: `fato_analise_outubro_rosa` (Q2 — Campanha)

**Granularidade:** Município × Ano × Mês da Campanha (Out/Nov/Dez).

**Fórmula (Especificação):** $\text{Variação Campanha (\%)} = \frac{\text{Exames Mês Associado} - \text{Exames Setembro}}{\text{Exames Setembro}} \times 100$

* Setembro é o período de referência do mesmo ano/município.
* Agregação prévia por (`codigo_municipio`, `ano`, `mes`) garante o join 1:1 mesmo se a Silver contiver múltiplas linhas por competência.

In [0]:
# ---------------------------------------------------------------------------
# 2. GOLD: fato_analise_outubro_rosa (Q2 — comparação com Setembro)
# ---------------------------------------------------------------------------
df_campanha_base = df_silver_exames.filter(F.col("mes").isin([9, 10, 11, 12]))

# Referência: exames de Setembro do mesmo ano/município (agregado → join 1:1)
df_setembro = (
    df_campanha_base
    .filter(F.col("mes") == 9)
    .groupBy("codigo_municipio", "ano")
    .agg(F.sum("quantidade").alias("exames_referencia_setembro"))
)

df_q2_campanha = (
    df_campanha_base
    .filter(F.col("mes").isin([10, 11, 12]))
    .groupBy("codigo_municipio", "municipio", "ano", "mes", "dt_competencia")
    .agg(F.sum("quantidade").alias("exames_mes"))
    .join(df_setembro, on=["codigo_municipio", "ano"], how="left")
    .withColumn(
        "ds_periodo_campanha",
        F.when(F.col("mes") == 10, F.lit("Mês da Campanha (Outubro)"))
         .when(F.col("mes") == 11, F.lit("Mês Subsequente 1 (Novembro)"))
         .otherwise(F.lit("Mês Subsequente 2 (Dezembro)")),
    )
    .withColumn(
        "variacao_campanha_pct",
        F.when(F.col("exames_referencia_setembro") > 0,
               F.round(((F.col("exames_mes") - F.col("exames_referencia_setembro"))
                        / F.col("exames_referencia_setembro")) * 100, 2)),
    )
    .withColumn("cd_ibge", F.lpad(F.col("codigo_municipio").cast("string"), 6, "0"))
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_SILVER}.exames_geral"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano", "mes", "dt_competencia",
        "ds_periodo_campanha", "exames_mes", "exames_referencia_setembro",
        "variacao_campanha_pct", "_silver_source", "_gold_processed_time",
    )
)

validar_e_gravar(
    df_q2_campanha,
    "fato_analise_outubro_rosa",
    pk_cols=["codigo_municipio", "ano", "mes"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "mes", "ds_periodo_campanha"],
)

display(df_q2_campanha.orderBy("codigo_municipio", "ano", "mes").limit(12))

## 3. GOLD: `fato_conformidade_faixa_etaria` (Q3)

**Granularidade:** Município × Ano × **Faixa Etária detalhada (5 em 5 anos)** — cada linha mantém também o `grupo_etario_diretriz` (3 agrupamentos macro) e as métricas do grupo (`quantidade_grupo`, `proporcao_grupo_pct`), permitindo filtrar por faixa quinquenal **ou** por grupo macro no dashboard.

**Categorias exigidas pelo TCC:**
* `<25 anos` — Sobrerrastreamento
* `25 a 64 anos` — População-Alvo (Ministério da Saúde)
* `≥65 anos` — Sub-rastreamento

> **Classificação robusta:** a Silver armazena faixas como `"10 a 14 anos"`, `"Até 9 anos"`, `"Acima de 79 anos"`. A classificação usa o limite inferior extraído numericamente (`regexp_extract`) + `fl_populacao_alvo`, cobrindo todos os 16 valores reais da Silver.

In [0]:
# ---------------------------------------------------------------------------
# 3. GOLD: fato_conformidade_faixa_etaria (Q3)
# ---------------------------------------------------------------------------
df_faixa = (
    spark.table(f"{CATALOG}.{SCHEMA_SILVER}.faixa_etaria")
    # Correção (31/out/2026): exclui a faixa 'Ignorado' (2021) — sem informação etária,
    # não se enquadra em nenhum dos 3 grupos da diretriz (sem o filtro cairia no
    # otherwise como '65 anos ou mais'). 'Até 9 anos' (2024/2025) permanece: idade
    # informada → classifica corretamente como sobrerastreamento (<25 anos).
    .filter(F.col("faixa_etaria") != "Ignorado")
)

# Classificação em 3 grupos da diretriz — extrai o limite inferior numérico da faixa
df_faixa_grupo = df_faixa.withColumn(
    "idade_limite_inferior",
    F.expr("try_cast(regexp_extract(faixa_etaria, '([0-9]+)', 1) as int)"),
).withColumn(
    "grupo_etario_diretriz",
    F.when(F.col("fl_populacao_alvo"), F.lit("25 a 64 anos (População-Alvo)"))
     .when(F.col("idade_limite_inferior") < 25, F.lit("<25 anos (Sobrerrastreamento)"))
     .otherwise(F.lit("65 anos ou mais (Sub-rastreamento)")),
)

win_muni_ano = Window.partitionBy("codigo_municipio", "ano")
win_grupo = Window.partitionBy("codigo_municipio", "ano", "grupo_etario_diretriz")

df_q3 = (
    df_faixa_grupo
    .groupBy("codigo_municipio", "municipio", "ano", "faixa_etaria", "grupo_etario_diretriz")
    .agg(F.sum("quantidade").alias("quantidade_exames"))
    # Total do município/ano (todas as faixas com informação etária)
    .withColumn("total_exames_muni_ano", F.sum("quantidade_exames").over(win_muni_ano))
    # Proporção da FAIXA DETALHADA (5 em 5 anos) dentro do município/ano
    .withColumn(
        "proporcao_pct",
        F.when(F.col("total_exames_muni_ano") > 0,
               F.round((F.col("quantidade_exames") / F.col("total_exames_muni_ano")) * 100, 2)),
    )
    # Métricas do GRUPO MACRO (3 agrupamentos da diretriz), repetidas nas faixas do grupo
    .withColumn("quantidade_grupo", F.sum("quantidade_exames").over(win_grupo))
    .withColumn(
        "proporcao_grupo_pct",
        F.when(F.col("total_exames_muni_ano") > 0,
               F.round((F.col("quantidade_grupo") / F.col("total_exames_muni_ano")) * 100, 2)),
    )
    .withColumn("cd_ibge", F.lpad(F.col("codigo_municipio").cast("string"), 6, "0"))
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_SILVER}.faixa_etaria"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano", "faixa_etaria",
        "grupo_etario_diretriz", "quantidade_exames", "total_exames_muni_ano",
        "proporcao_pct", "quantidade_grupo", "proporcao_grupo_pct",
        "_silver_source", "_gold_processed_time",
    )
)

validar_e_gravar(
    df_q3,
    "fato_conformidade_faixa_etaria",
    pk_cols=["codigo_municipio", "ano", "faixa_etaria"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "faixa_etaria", "grupo_etario_diretriz"],
)

display(df_q3.orderBy("codigo_municipio", "ano", "faixa_etaria").limit(20))

### 3b. GOLD: `fato_indicador_populacao_alvo` (Q3 — KPI isolado)

**Granularidade:** Município × Ano (1 linha por município/ano — ideal para cartões KPI no Power BI / AI-BI Dashboard).

**Fórmula (Especificação):** $\text{Proporção na população-alvo (\%)} = \frac{\text{Exames de 25 a 64 anos}}{\text{Total de exames com informação etária}} \times 100$

Além do KPI principal (25–64 anos), a tabela traz as proporções de sobrerrastreamento (<25) e sub-rastreamento (≥65) em colunas separadas na mesma linha.

In [0]:
# ---------------------------------------------------------------------------
# 3b. GOLD: fato_indicador_populacao_alvo (Q3 — KPI da população-alvo)
# ---------------------------------------------------------------------------
GRUPO_ALVO = "25 a 64 anos (População-Alvo)"
GRUPO_SOBRE = "<25 anos (Sobrerrastreamento)"
GRUPO_SUB = "65 anos ou mais (Sub-rastreamento)"

df_indicador_alvo = (
    df_q3
    .groupBy("codigo_municipio", "municipio", "ano")
    .agg(
        F.max("total_exames_muni_ano").alias("total_exames_informacao_etaria"),
        F.max(F.when(F.col("grupo_etario_diretriz") == GRUPO_ALVO, F.col("quantidade_grupo"))).alias("exames_populacao_alvo"),
        F.max(F.when(F.col("grupo_etario_diretriz") == GRUPO_SOBRE, F.col("quantidade_grupo"))).alias("exames_sobrerastreamento"),
        F.max(F.when(F.col("grupo_etario_diretriz") == GRUPO_SUB, F.col("quantidade_grupo"))).alias("exames_subrastreamento"),
    )
    .withColumn(
        "proporcao_populacao_alvo_pct",
        F.when(F.col("total_exames_informacao_etaria") > 0,
               F.round((F.col("exames_populacao_alvo") / F.col("total_exames_informacao_etaria")) * 100, 2)),
    )
    .withColumn(
        "proporcao_sobrerastreamento_pct",
        F.when(F.col("total_exames_informacao_etaria") > 0,
               F.round((F.col("exames_sobrerastreamento") / F.col("total_exames_informacao_etaria")) * 100, 2)),
    )
    .withColumn(
        "proporcao_subrastreamento_pct",
        F.when(F.col("total_exames_informacao_etaria") > 0,
               F.round((F.col("exames_subrastreamento") / F.col("total_exames_informacao_etaria")) * 100, 2)),
    )
    .withColumn("cd_ibge", F.lpad(F.col("codigo_municipio").cast("string"), 6, "0"))
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_SILVER}.faixa_etaria"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano",
        "exames_populacao_alvo", "exames_sobrerastreamento", "exames_subrastreamento",
        "total_exames_informacao_etaria",
        "proporcao_populacao_alvo_pct", "proporcao_sobrerastreamento_pct",
        "proporcao_subrastreamento_pct", "_silver_source", "_gold_processed_time",
    )
)

validar_e_gravar(
    df_indicador_alvo,
    "fato_indicador_populacao_alvo",
    pk_cols=["codigo_municipio", "ano"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano"],
)

display(df_indicador_alvo.orderBy("codigo_municipio", "ano"))

## 4. GOLD: `fato_adequabilidade_amostras` (Q4)

**Granularidade:** Município × Ano × Categoria de Adequabilidade (Satisfatória / Insatisfatória / Rejeitada).

**Fórmula (Especificação):** $\text{Amostras Insatisfatórias (\%)} = \frac{\text{Amostras Insatisfatórias}}{\text{Total de Amostras com Adequabilidade}} \times 100$

In [0]:
# ---------------------------------------------------------------------------
# 4. GOLD: fato_adequabilidade_amostras (Q4)
# ---------------------------------------------------------------------------
df_adeq = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.adequabilidade")

win_adeq_total = Window.partitionBy("codigo_municipio", "ano")

df_q4 = (
    df_adeq
    .groupBy("codigo_municipio", "municipio", "ano", "categoria_adequabilidade")
    .agg(F.sum("quantidade").alias("quantidade_amostras"))
    .withColumn("total_amostras_muni_ano", F.sum("quantidade_amostras").over(win_adeq_total))
    .withColumn(
        "proporcao_adequabilidade_pct",
        F.when(F.col("total_amostras_muni_ano") > 0,
               F.round((F.col("quantidade_amostras") / F.col("total_amostras_muni_ano")) * 100, 2)),
    )
    .withColumn("cd_ibge", F.lpad(F.col("codigo_municipio").cast("string"), 6, "0"))
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_SILVER}.adequabilidade"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano", "categoria_adequabilidade",
        "quantidade_amostras", "total_amostras_muni_ano", "proporcao_adequabilidade_pct",
        "_silver_source", "_gold_processed_time",
    )
)

validar_e_gravar(
    df_q4,
    "fato_adequabilidade_amostras",
    pk_cols=["codigo_municipio", "ano", "categoria_adequabilidade"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "categoria_adequabilidade"],
)

display(df_q4.orderBy("codigo_municipio", "ano", "categoria_adequabilidade"))

## 5. GOLD: `fato_exames_alterados` (Q5)

**Granularidade:** Município × Ano × Faixa Etária.

**Fórmula (Especificação):** $\text{Exames Alterados (\%)} = \frac{\text{Exames Alterados (colo)}}{\text{Total de Exames da Faixa Etária}} \times 100$

* Numerador: `silver.resultados_citopatologicos` (`grupo_resultado` = exames citopatológicos de colo **alterados** por faixa etária).
* Denominador: `silver.faixa_etaria` — total de exames por município/ano/faixa etária.
* Faixas sem exames no denominador ficam com proporção `NULL` (sem divisão por zero fabricada).
* **Correção (31/out/2026):** a junção com o denominador é `FULL OUTER` — faixas do relatório de faixa etária sem registro de alterados (colunas zeradas omitidas pelo TABNET, ex.: "10 a 14" em 2024/25) entram com `quantidade_alterados = 0`, e o denominador fecha exatamente com o total de exames da regional (131.058).

> **Nota metodológica (critérios de inclusão/exclusão do denominador):** a fórmula oficial do TCC usa como denominador o *total de exames satisfatórios com resultado válido*, **excluindo** amostras rejeitadas/insatisfatórias. Na fonte SISCAN, porém, o relatório por faixa etária não segmenta por categoria de adequabilidade, e o relatório de adequabilidade não segmenta por idade — **não há cruzamento possível** entre idade e adequabilidade. O denominador operacional (`silver.faixa_etaria` = total de exames com idade preenchida) inclui, portanto, amostras insatisfatórias/rejeitadas, o que torna a proporção de exames alterados uma **estimativa conservadora (subestimada)**. Esta limitação deve ser declarada na seção de limitações metodológicas do TCC.

In [0]:
# ---------------------------------------------------------------------------
# 5. GOLD: fato_exames_alterados (Q5)
# ---------------------------------------------------------------------------
df_resultados = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.resultados_citopatologicos")
df_exames_totais = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.faixa_etaria")

# Numerador: exames alterados por município/ano/faixa (agregado → join 1:1)
df_alterados = (
    df_resultados
    .groupBy("codigo_municipio", "municipio", "ano", "faixa_etaria")
    .agg(F.sum("quantidade").alias("quantidade_alterados"))
)

# Denominador: total de exames por município/ano/faixa
# (municipio entra no groupBy para viabilizar o full_outer abaixo — coluna obrigatória)
df_denominador = (
    df_exames_totais
    .groupBy("codigo_municipio", "municipio", "ano", "faixa_etaria")
    .agg(F.sum("quantidade").alias("total_exames_validos"))
)

# Correção (31/out/2026): FULL OUTER JOIN entre numerador e denominador.
# O LEFT join anterior descartava faixas presentes no relatório de faixa etária
# mas sem coluna no relatório de alterados (TABNET omite colunas zeradas:
# '10 a 14' em 2024/25, 'Acima de 79' em 2025, 'Até 9 anos' e 'Ignorado'),
# deixando o denominador em 130.987. Com o full outer + coalesce(0) no numerador,
# a tabela cobre a grade completa de faixas por município/ano e o denominador
# fecha exatamente com o total de exames da regional (131.058).
df_q5 = (
    df_alterados.alias("a")
    .join(df_denominador.alias("d"), on=["codigo_municipio", "ano", "faixa_etaria"], how="full_outer")
    .select(
        F.col("codigo_municipio"),
        F.coalesce(F.col("a.municipio"), F.col("d.municipio")).alias("municipio"),
        F.col("ano"),
        F.col("faixa_etaria"),
        F.coalesce(F.col("a.quantidade_alterados"), F.lit(0)).alias("quantidade_alterados"),
        F.col("d.total_exames_validos"),
    )
    .withColumn(
        "proporcao_alterados_pct",
        F.when(F.col("total_exames_validos") > 0,
               F.round((F.col("quantidade_alterados") / F.col("total_exames_validos")) * 100, 2)),
    )
    .withColumn("cd_ibge", F.lpad(F.col("codigo_municipio").cast("string"), 6, "0"))
    .withColumn("_silver_source", F.lit(
        f"{CATALOG}.{SCHEMA_SILVER}.resultados_citopatologicos + {CATALOG}.{SCHEMA_SILVER}.faixa_etaria"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano", "faixa_etaria",
        "quantidade_alterados", "total_exames_validos", "proporcao_alterados_pct",
        "_silver_source", "_gold_processed_time",
    )
)

validar_e_gravar(
    df_q5,
    "fato_exames_alterados",
    pk_cols=["codigo_municipio", "ano", "faixa_etaria"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "faixa_etaria"],
)

display(df_q5.orderBy("codigo_municipio", "ano", "faixa_etaria").limit(20))

## 6. Validação Final das Tabelas Gold

Verificação pós-gravação lendo as tabelas diretamente do Unity Catalog:
* Contagem de linhas de cada tabela fato
* Soma de `participacao_territorial_pct` por competência ≈ 100 (Q1/Q2)
* Soma de `proporcao_pct` por município/ano ≈ 100 (Q3 e Q4)
* Soma das 3 proporções do KPI de população-alvo ≈ 100 (`fato_indicador_populacao_alvo`)
* Escopo: 11 municípios da 10ª CRS, anos 2020–2025

In [0]:
# ---------------------------------------------------------------------------
# 6. Validação final — lê as tabelas Gold gravadas no Unity Catalog
# ---------------------------------------------------------------------------
tabelas_gold = [
    "fato_exames_temporais_territoriais",
    "fato_analise_outubro_rosa",
    "fato_conformidade_faixa_etaria",
    "fato_indicador_populacao_alvo",
    "fato_adequabilidade_amostras",
    "fato_exames_alterados",
]

print("=== Contagem de linhas (Gold no Unity Catalog) ===")
for t in tabelas_gold:
    df_t = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.{t}")
    print(f"  {t}: {df_t.count()} linhas")

# 1. Q1/Q2: participação territorial soma ~100 por competência
df_g1 = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.fato_exames_temporais_territoriais")
soma_part = (
    df_g1.groupBy("dt_competencia")
    .agg(F.round(F.sum("participacao_territorial_pct"), 1).alias("soma_participacao"))
)
err_part = soma_part.filter(~F.col("soma_participacao").between(99.5, 100.5)).count()
print(f"\nQ1/Q2 — competências fora de [99.5, 100.5]% de participação somada: {err_part}")

# 2. Q3: proporções somam ~100 por município/ano
df_g3 = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.fato_conformidade_faixa_etaria")
err_q3 = (
    df_g3.groupBy("codigo_municipio", "ano")
    .agg(F.round(F.sum("proporcao_pct"), 1).alias("s"))
    .filter(~F.col("s").between(99.5, 100.5))
    .count()
)
print(f"Q3 — pares município/ano fora de [99.5, 100.5]%: {err_q3}")

# 2b. Q3-KPI: proporções da população-alvo somam ~100 por município/ano
df_ind = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.fato_indicador_populacao_alvo")
err_ind = (
    df_ind.withColumn(
        "soma",
        F.round(
            F.col("proporcao_populacao_alvo_pct")
            + F.col("proporcao_sobrerastreamento_pct")
            + F.col("proporcao_subrastreamento_pct"), 1),
    )
    .filter(~F.col("soma").between(99.5, 100.5))
    .count()
)
print(f"Q3-KPI — município/ano fora de [99.5, 100.5]%: {err_ind}")

# 3. Q4: proporções somam ~100 por município/ano
df_g4 = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.fato_adequabilidade_amostras")
err_q4 = (
    df_g4.groupBy("codigo_municipio", "ano")
    .agg(F.round(F.sum("proporcao_adequabilidade_pct"), 1).alias("s"))
    .filter(~F.col("s").between(99.5, 100.5))
    .count()
)
print(f"Q4 — pares município/ano fora de [99.5, 100.5]%: {err_q4}")

# 4. Escopo territorial e temporal
escopo = df_g1.agg(
    F.countDistinct("codigo_municipio").alias("municipios"),
    F.min("ano").alias("ano_min"),
    F.max("ano").alias("ano_max"),
)
display(escopo)

assert err_part == 0 and err_q3 == 0 and err_ind == 0 and err_q4 == 0, "❌ Validação final falhou — verifique as contagens acima."
print("\n✅ Validação final da Camada Gold concluída com sucesso.")